In [5]:
import torch
import numpy as np
import re
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# Load the model once
print("Loading model...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print("Model loaded successfully!")


Loading model...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6595.82it/s]


Model loaded successfully!


In [6]:
def clean_text(text):
    """Removes Markdown symbols and formatting noise"""
    text = re.sub(r'#+', '', text)        # Remove # headers
    text = re.sub(r'\*\*', '', text)      # Remove ** bolding
    text = re.sub(r'\*', '', text)        # Remove * bullets
    text = text.replace('\n', ' ')       # Replace newlines with spaces
    text = re.sub(r'\s+', ' ', text)     # Remove double spaces
    return text.strip()

def encode_cleaned_text(cleaned_text, model):
    """Takes ALREADY CLEANED text and converts it to a document vector"""
    # Split into sentences to avoid the 256-token truncation limit
    sentences = [s.strip() for s in cleaned_text.split('.') if len(s.strip()) > 0]
    
    # Encode each sentence separately
    sentence_embeddings = model.encode(sentences)
    
    # Average all sentence vectors into one "Document Vector"
    return np.mean(sentence_embeddings, axis=0)

print("Logic functions ready!")


Logic functions ready!


In [7]:
text_1_raw = """
**Sheikh Hasina** is a Bangladeshi politician who served as the **Prime Minister of Bangladesh** for three consecutive terms (1996–2001 and 2009–2024), making her the longest-serving female head of government in the country's history.\n\nHere are the key details about her life and career:\n\n### 1. Family Background\nShe is the daughter of **Sheikh Mujibur Rahman**, the founding father of Bangladesh and its first president. Her father is revered as \"Bangabandhu\" (Friend of Bengal) for his pivotal role in the liberation of Bangladesh from Pakistan in 1971.\n\n### 2. Political Leadership\nHasina is the President of the **Awami League**, one of the oldest and largest political parties in Bangladesh. Her political career has been defined by a long-standing rivalry with Khaleda Zia (the leader of the Bangladesh Nationalist Party, or BNP).\n\n### 3. Tenure as Prime Minister\nDuring her time in power, particularly from 2009 to 2024, her administration was characterized by:\n*   **Economic Growth:** Bangladesh saw significant GDP growth, improvements in infrastructure (such as the Padma Bridge), and a rise in the standard of living.\n*   **Secularism:** She positioned herself as a defender of secularism and a combatant against religious extremism.\n*   **Controversy:** Her later years in power were marked by accusations of authoritarianism, the suppression of political opposition, and human rights abuses.\n\n### 4. Recent Events (August 2024)\nIn the summer of 2024, Sheikh Hasina's government faced massive protests led primarily by students over a controversial quota system for government jobs. The protests escalated into a nationwide uprising against her administration.\n\nOn **August 5, 2024**, following a massive march on the capital city of Dhaka and the threat of a military intervention, **Sheikh Hasina resigned from her position as Prime Minister and fled the country**, flying to India.\n\n### Summary\nSheikh Hasina is a polarizing figure: to her supporters, she is the leader who modernized Bangladesh and honored her father's legacy; to her critics, she is a leader who dismantled democratic norms to maintain power.
"""

text_2_raw = """
**Ziaur Rahman** (often referred to as Ziaur Rahman, though he rose to the rank of Army General) was a pivotal military and political figure in the history of Bangladesh. He served as the **President of Bangladesh** from 1977 to 1981.\n\nDepending on the context of your question, he is viewed in three primary roles:\n\n### 1. The Military Leader (The \"Major\" and General)\nZiaur Rahman was a soldier in the Pakistan Army who joined the Bangladesh Liberation War in 1971. He became a national hero during the war after broadcasting the declaration of independence on behalf of Sheikh Mujibur Rahman on March 26, 1971. His leadership during the war earned him immense respect within the military and among the general public.\n\n### 2. The Political Leader and President\nAfter the assassination of Sheikh Mujibur Rahman in 1975 and a period of intense political instability and military coups, Ziaur Rahman eventually emerged as the Chief Martial Law Administrator and later the President.\n\n*   **Shift in Ideology:** He moved Bangladesh away from the \"socialist\" economic model of the early 1970s toward a more market-oriented economy.\n*   **Multi-Party Democracy:** He shifted the country from a one-party system back to a multi-party system.\n*   **Founding the BNP:** In 1978, he founded the **Bangladesh Nationalist Party (BNP)**, which remains one of the two dominant political parties in Bangladesh today.\n*   **Nationalism:** He emphasized \"Bangladeshi Nationalism,\" which distinguished the identity of Bangladesh from the ethnic \"Bengali Nationalism\" (which included West Bengal in India).\n\n### 3. His Legacy and Controversy\nZiaur Rahman is a polarizing figure in Bangladeshi history:\n*   **Supporters** view him as a visionary who stabilized the economy, modernized agriculture, and fostered a distinct national identity.\n*   **Critics** point to his rise to power through military rule and argue that his policies laid the groundwork for the politicization of the military.\n\n**Death:**\nPresident Ziaur Rahman was assassinated on May 30, 1981, during a failed military coup attempt in Chittagong.\n\n***Note on the title \"Major\":*** *While he started the liberation war as a Major, he is most commonly referred to in historical and political texts as **President Zia** or **General Zia**, as he attained those ranks during his leadership of the country.*
"""

# APPLY CLEANING
cleaned_1 = clean_text(text_1_raw)
cleaned_2 = clean_text(text_2_raw)

print("--- CLEANED TEXT 1 ---")
print(cleaned_1)
print("\n" + "="*50 + "\n")
print("--- CLEANED TEXT 2 ---")
print(cleaned_2)


--- CLEANED TEXT 1 ---
Sheikh Hasina is a Bangladeshi politician who served as the Prime Minister of Bangladesh for three consecutive terms (1996–2001 and 2009–2024), making her the longest-serving female head of government in the country's history. Here are the key details about her life and career: 1. Family Background She is the daughter of Sheikh Mujibur Rahman, the founding father of Bangladesh and its first president. Her father is revered as "Bangabandhu" (Friend of Bengal) for his pivotal role in the liberation of Bangladesh from Pakistan in 1971. 2. Political Leadership Hasina is the President of the Awami League, one of the oldest and largest political parties in Bangladesh. Her political career has been defined by a long-standing rivalry with Khaleda Zia (the leader of the Bangladesh Nationalist Party, or BNP). 3. Tenure as Prime Minister During her time in power, particularly from 2009 to 2024, her administration was characterized by: Economic Growth: Bangladesh saw signifi

In [8]:
# 1. Generate embeddings using the cleaned versions
emb1 = encode_cleaned_text(cleaned_1, model)
emb2 = encode_cleaned_text(cleaned_2, model)

# 2. Reshape for cosine similarity
vec1 = emb1.reshape(1, -1)
vec2 = emb2.reshape(1, -1)

# 3. Calculate
sim = cosine_similarity(vec1, vec2)[0][0]

print("-" * 30)
print(f"The Cosine Similarity is: {sim:.6f}")
print("-" * 30)

if sim > 0.90:
    print("Result: The texts are almost identical in meaning.")
elif sim > 0.70:
    print("Result: The texts are very similar.")
elif sim > 0.50:
    print("Result: The texts are moderately related.")
else:
    print("Result: The texts are quite different.")


------------------------------
The Cosine Similarity is: 0.642133
------------------------------
Result: The texts are moderately related.
